# Prática: leitura de hidrômetros (Luminus, Grupo 2)

Notebook fino: toda a lógica está em `src/hidrometros/` (instalável com `pip install -e .`). Executado de ponta a ponta sobre os artefatos locais; as saídas têm só tabelas e texto. A galeria de erros da seção 5 desenha recortes do dataset, então sua saída é limpa antes do commit (nenhuma imagem do dataset no git). Para reproduzir do zero, rode no terminal `hidrometros prep` e `caffeinate -i hidrometros train all --resume` (ver README).

## 1. Dados

In [1]:
import pandas as pd
from hidrometros import data

print(data.check_integrity())
ann = data.load_annotations()
ann.head()

IntegrityReport(n_rows=1244, n_images=1244, n_masks=1244, n_collage=1244, n_unreadable_images=0, n_unreadable_masks=0, missing_images=0, missing_masks=0, splits_sha256='ca4c5d8ad131d63eae3a8acd5bd6ac1091f55e02426d8ad083af98cfa65abf2b', splits_ok=True)


,photo_name,value,location,image_id,label,split
0,id_53_value_595_825.jpg,595.825,"{'type': 'polygon', 'data': [{'x': 0.30788, 'y...",53,00595825,train
1,id_553_value_65_475.jpg,65.475,"{'type': 'polygon', 'data': [{'x': 0.26133, 'y...",553,00065475,train
2,id_407_value_21_86.jpg,21.860,"{'type': 'polygon', 'data': [{'x': 0.27545, 'y...",407,00021860,test
3,id_252_value_313_322.jpg,313.322,"{'type': 'polygon', 'data': [{'x': 0.21967, 'y...",252,00313322,train
4,id_851_value_305_162.jpg,305.162,"{'type': 'polygon', 'data': [{'x': 0.06983, 'y...",851,00305162,train


## 2. Preparação: retificação, orientação, 8 recortes de 48x64

In [2]:
# Estatísticas da preparação (geradas por `hidrometros prep`; ~3 min, não repetidas aqui).
import json
st = json.loads((data.ROOT / 'reports' / 'prep_stats.json').read_text())
resumo = {k: st[k] for k in ['n_annotated', 'n_valid', 'n_crops', 'images_per_split', 'crops_per_split',
                              'rotated_by_geometry', 'flipped_final', 'images_in_multiple_splits']}
resumo['descartadas'] = st['discarded']
resumo

{'n_annotated': 1244,
 'n_valid': 1243,
 'n_crops': 9944,
 'images_per_split': {'train': 870, 'test': 187, 'val': 186},
 'crops_per_split': {'train': 6960, 'test': 1496, 'val': 1488},
 'rotated_by_geometry': 136,
 'flipped_final': 56,
 'images_in_multiple_splits': 0,
 'descartadas': ['id_664_value_45_2676.jpg']}

## 3. Treino dos quatro modelos (mesmo protocolo)

In [3]:
# Histórico de treino dos quatro modelos (gerado por `hidrometros train all`; não retreinamos aqui).
rows = []
for m in ['smallcnn', 'alexnet', 'vgg16_bn', 'resnet18']:
    h = pd.read_csv(data.ROOT / f'history_{m}.csv')
    rows.append({'model': m, 'epocas_rodadas': len(h), 'melhor_epoca': int(h.loc[h.val_f1.idxmax(), 'epoch']),
                 'val_f1_max': round(float(h.val_f1.max()), 4), 'val_acc_ultima': round(float(h.val_acc.iloc[-1]), 4),
                 'segundos_totais': round(float(h.seconds.sum()), 1)})
pd.DataFrame(rows)

,model,epocas_rodadas,melhor_epoca,val_f1_max,val_acc_ultima,segundos_totais
0,smallcnn,10,10,0.7853,0.8253,89.4
1,alexnet,8,8,0.9152,0.9328,286.8
2,vgg16_bn,8,6,0.9365,0.9422,3530.4
3,resnet18,8,6,0.9606,0.9651,444.7


## 4. Avaliação no teste: results.csv, relatórios, figuras

In [4]:
# Resultados no conjunto de teste (results.csv) e do desafio extra (results_extra.csv).
# As figuras estão em figures/ e embutidas no README (geradas por `hidrometros figures`).
display(pd.read_csv(data.ROOT / 'results.csv'))
display(pd.read_csv(data.ROOT / 'results_extra.csv'))

,model,acc_digit,f1_macro,acc_reading,params_M,s_per_epoch,ms_per_image,ckpt_MB,best_epoch
0,smallcnn,0.8155,0.7711,0.3102,0.34,7.0,1.42,1.4,10
1,alexnet,0.9412,0.9261,0.6952,57.04,35.0,6.86,228.2,8
2,vgg16_bn,0.9532,0.9455,0.7701,134.31,473.6,20.80,537.3,6
3,resnet18,0.9699,0.9654,0.8556,11.18,57.4,6.42,44.8,6


,model,acc_digit,f1_macro,acc_reading,params_M,s_per_epoch,ms_per_image,ckpt_MB,best_epoch
0,smallcnn,0.7527,0.6988,0.1818,0.34,7.0,1.42,1.4,10
1,alexnet,0.9218,0.9040,0.6417,57.04,30.7,5.99,228.2,7
2,vgg16_bn,0.9425,0.9346,0.7594,134.31,446.1,22.64,537.3,7
3,resnet18,0.9566,0.9505,0.8128,11.18,51.2,6.53,44.8,6


## 5. Galeria dos 16 erros mais confiantes por modelo (apenas local)

A célula abaixo desenha recortes do dataset. Foi executada localmente e a saída foi **limpa antes do commit** (`jupyter nbconvert --clear-output`), porque as imagens do dataset não podem ser versionadas.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from hidrometros import data

idx = pd.read_csv(data.ROOT / 'digits' / 'index.csv')
for MODEL in ['smallcnn', 'alexnet', 'vgg16_bn', 'resnet18']:
    p = pd.read_csv(data.ROOT / 'reports' / f'predictions_{MODEL}.csv')
    err = p[p.y_true != p.y_pred].nlargest(16, 'confidence').merge(idx, on=['photo_name', 'position'])
    fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (_, r) in zip(axes.ravel(), err.iterrows()):
        ax.imshow(Image.open(data.ROOT / r.crop_path))
        ax.set_title(f'{r.y_true}->{r.y_pred} ({r.confidence:.2f})', fontsize=8)
    fig.suptitle(f'{MODEL}: 16 erros mais confiantes (teste)')
    plt.show()
